# Cache Benchmark Visualization

Loads `benchmark_results.csv` (unit) and `benchmark_integration_results.csv` (integration), then renders pivot tables and grouped bar charts for hit rate, latency, and throughput across (policy × workload × capacity).

Run from `src/kvstore/`. Run `benchmark.py` first (in either or both modes) to populate the CSVs.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

HERE = Path(".")

SOURCES = [
    ("unit", "benchmark_results.csv"),
    ("integration", "benchmark_integration_results.csv"),
]

dfs: dict[str, pd.DataFrame] = {}
for label, fname in SOURCES:
    path = HERE / fname
    if path.exists():
        dfs[label] = pd.read_csv(path)
    else:
        print(f"skip {label}: {path} not found")

for mode, df in dfs.items():
    print(
        f"{mode}: {len(df)} rows | "
        f"policies={sorted(df.policy.unique())} | "
        f"workloads={sorted(df.workload.unique())} | "
        f"capacities={sorted(df.capacity.unique())}"
    )

## Pivot tables

Rows: `(workload, capacity)`. Columns: `policy`. Cells: the metric value.

In [ ]:
def pivot(df: pd.DataFrame, metric: str, decimals: int) -> pd.DataFrame:
    return (
        df.pivot_table(index=["workload", "capacity"], columns="policy", values=metric)
        .round(decimals)
    )


METRIC_DECIMALS = [
    ("hit_rate", 4),
    ("avg_latency_us", 2),
    ("p99_latency_us", 2),
    ("throughput_ops_per_sec", 0),
]

for mode, df in dfs.items():
    for metric, dec in METRIC_DECIMALS:
        print(f"=== {mode}: {metric} ===")
        display(pivot(df, metric, dec))
    print()

## Bar charts

One figure per `(mode, metric)`. Each figure has one subplot per capacity; bars are grouped by policy with workloads on the x-axis.

In [ ]:
def grouped_bar(df: pd.DataFrame, metric: str, capacity: int, ax) -> None:
    sub = df[df["capacity"] == capacity]
    p = sub.pivot_table(index="workload", columns="policy", values=metric)
    p.plot(kind="bar", ax=ax, edgecolor="black", linewidth=0.4)
    ax.set_title(f"capacity={capacity}")
    ax.set_ylabel(metric)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
    ax.legend(title="policy", fontsize=8, ncol=2, loc="best")
    ax.grid(axis="y", alpha=0.3)


CHART_METRICS = [
    ("hit_rate", "hit rate"),
    ("p99_latency_us", "p99 latency (us)"),
    ("avg_latency_us", "avg latency (us)"),
    ("throughput_ops_per_sec", "throughput (ops/s)"),
]

for metric, label in CHART_METRICS:
    for mode, df in dfs.items():
        caps = sorted(df["capacity"].unique())
        ncols = len(caps)
        fig, axes = plt.subplots(
            1, ncols, figsize=(6 * ncols, 4.5), squeeze=False, sharey=True
        )
        for ax, cap in zip(axes[0], caps):
            grouped_bar(df, metric, cap, ax)
        fig.suptitle(f"{mode}: {label}", fontsize=14)
        fig.tight_layout()
        plt.show()

## Hit rate vs. p99 latency tradeoff

A scatter view: each point is one (policy, workload, capacity) combination. Colored by policy, marker shape by workload. Useful for spotting policies that buy hits at the cost of latency (or vice versa).

In [ ]:
import itertools

MARKERS = ["o", "s", "D", "^", "v", "P", "X", "*", "<", ">"]

for mode, df in dfs.items():
    fig, ax = plt.subplots(figsize=(9, 6))
    workloads = sorted(df["workload"].unique())
    policies = sorted(df["policy"].unique())
    color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    color_for = {p: color_cycle[i % len(color_cycle)] for i, p in enumerate(policies)}
    marker_for = {w: MARKERS[i % len(MARKERS)] for i, w in enumerate(workloads)}

    for (policy, workload), grp in df.groupby(["policy", "workload"]):
        ax.scatter(
            grp["hit_rate"], grp["p99_latency_us"],
            color=color_for[policy], marker=marker_for[workload],
            s=60, alpha=0.85, edgecolor="black", linewidth=0.4,
        )

    policy_handles = [
        plt.Line2D([], [], marker="o", linestyle="", color=color_for[p], label=p)
        for p in policies
    ]
    workload_handles = [
        plt.Line2D([], [], marker=marker_for[w], linestyle="", color="gray", label=w)
        for w in workloads
    ]
    legend1 = ax.legend(handles=policy_handles, title="policy", loc="upper left", fontsize=8)
    ax.add_artist(legend1)
    ax.legend(handles=workload_handles, title="workload", loc="lower right", fontsize=8)

    ax.set_xlabel("hit rate")
    ax.set_ylabel("p99 latency (us)")
    ax.set_title(f"{mode}: hit rate vs p99 latency")
    ax.grid(alpha=0.3)
    plt.show()